In [ ]:
import workshop_setup

In [2]:
from pathlib import Path
from maize.core.workflow import Workflow
from maize.core.node import Node
from maize.steps.io import LoadData, LogResult, Void, FileParameter, Return
from maize.steps.mai.molecule import LoadSmiles, LoadMolecule, Gypsum
from maize.core.interface import Input, Output
from maize.utilities.chem import IsomerCollection
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem.Draw import IPythonConsole
import pandas as pd
from pandas import DataFrame

In [ ]:
for desc in Descriptors._descList:
    print(desc)

In [4]:
class CalcDesc(Node):
    out: Output[pd.DataFrame] = Output()
    inp: Input[list[str]] = Input()

    def run(self):
        res = pd.DataFrame({'smiles': self.inp.receive()})
        res['ROMol'] = res['smiles'].apply(Chem.MolFromSmiles)
        res['MolWt'] = res['ROMol'].apply(Descriptors.MolWt)
        res['MolLogP'] = res['ROMol'].apply(Descriptors.MolLogP)
        return self.out.send(res)

In [5]:
mols = ['CCC', 'CCOC', 'CCCC']

In [6]:
flow = Workflow()
load = flow.add(LoadData[list[str]])
calc = flow.add(CalcDesc)
res = flow.add(Return[pd.DataFrame])
flow.connect(load.out, calc.inp)
flow.connect(calc.out, res.inp)
load.data.set(mols)
flow.check()

In [ ]:
flow.visualize()

In [ ]:
flow.execute()

In [ ]:
res.get()

In [10]:
from maize.utilities.chem.chem import IsomerCollection
iso1 = IsomerCollection.from_smiles('CCC(O)CN')

In [ ]:
iso1

In [ ]:
iso1.molecules[0]._molecule

In [ ]:
iso1.molecules[1]._molecule

In [14]:
class Smi2Mols(Node):
    inp: Input[list[str]] = Input()
    out: Output[list[IsomerCollection]] = Output()

    def run(self) -> None:
        smiles_list = self.inp.receive()
        mols = [IsomerCollection.from_smiles(smi) for smi in smiles_list]
        self.out.send(mols)

In [15]:
flow = Workflow()
load = flow.add(LoadData[list[str]])
emb = flow.add(Smi2Mols)
ret = flow.add(Return[list[IsomerCollection]])
flow.connect(load.out, emb.inp)
flow.connect(emb.out, ret.inp)
load.data.set(['CC', 'CCC(O)CN'])
flow.check()

In [ ]:
flow.execute()

In [17]:
results = ret.get()

In [ ]:
results